In [1]:
from typing import TypedDict

from dotenv import load_dotenv
load_dotenv()


True

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("google_genai:gemini-3.8-flash")

# llm.invoke("Who was the first person to walk on moon")






In [ ]:

from typing import Annotated
from langgraph.graph import StateGraph, START, END, add_messages


class State(TypedDict):
    messages: Annotated[list,add_messages]

def chatbot(state:State) -> State:
    return{"messages":[llm.invoke(state["messages"])]}

builder = StateGraph(State)

builder.add_node("chatbot_node",chatbot)
builder.add_edge(START,"chatbot_node")
builder.add_edge("chatbot_node",END)

graph = builder.compile()

In [ ]:
message={"role":"user","content":"Who walked on the moon for the first time? Print only the name"}
response =graph.invoke({"messages":[message]})
response["messages"]

In [ ]:
state = None

while True:
    in_message = input("You: ")
    if in_message.lower() in ("quit","exit"):
        break
    if state is None:
        state: State = {
            "messages":[{"role":"user","content":in_message}]
        }
    else:
        state["messages"].append({"role":"user","content":in_message})

    state = graph.invoke(state)
    print("Bot: ",state["messages"][-1].content)